# 🔬 Advanced Analytics ETL v3
ABC Pareto 分類、品牌績效、折扣 ROI、時間熱力圖、付款方式、地區分析、CLV 預測

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("3C_ETL_v3_extras") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

pg_url = "jdbc:postgresql://pgvector:5432/postgres"
pg_props = {"user": "postgres", "password": "admin", "driver": "org.postgresql.Driver"}

users = spark.read.jdbc(url=pg_url, table="users", properties=pg_props)
products = spark.read.jdbc(url=pg_url, table="products", properties=pg_props)
orders = spark.read.jdbc(url=pg_url, table="orders", properties=pg_props)
order_items = spark.read.jdbc(url=pg_url, table="order_items", properties=pg_props)
returns = spark.read.jdbc(url=pg_url, table="returns", properties=pg_props)

sales = order_items.join(orders, "order_id") \
    .join(products, "product_id") \
    .filter(col("status") == "Completed") \
    .withColumn("line_revenue", col("quantity") * col("unit_price")) \
    .withColumn("line_cost", col("quantity") * col("cost_price")) \
    .withColumn("line_profit", col("line_revenue") - col("line_cost"))

print("Data loaded. Sales rows:", sales.count())

Data loaded. Sales rows: 5078


### 1. ABC Pareto 分類 (80/20 法則)

In [2]:
product_rev = sales.groupBy("product_id", "brand", "name", "category") \
    .agg(sum("line_revenue").alias("total_revenue"), sum("quantity").alias("total_units"))

total = product_rev.agg(sum("total_revenue")).collect()[0][0]

w = Window.orderBy(desc("total_revenue"))
gold_abc = product_rev \
    .withColumn("revenue_rank", row_number().over(w)) \
    .withColumn("cumulative_revenue", sum("total_revenue").over(w.rowsBetween(Window.unboundedPreceding, 0))) \
    .withColumn("cumulative_pct", round(col("cumulative_revenue") / lit(total) * 100, 2)) \
    .withColumn("abc_class",
        when(col("cumulative_pct") <= 80, "A")
        .when(col("cumulative_pct") <= 95, "B")
        .otherwise("C")) \
    .withColumn("revenue_pct", round(col("total_revenue") / lit(total) * 100, 2))

print("ABC Analysis ready. Class distribution:")
gold_abc.groupBy("abc_class").agg(count("*").alias("products"), sum("total_revenue").alias("rev")).show()

ABC Analysis ready. Class distribution:
+---------+--------+----------+
|abc_class|products|       rev|
+---------+--------+----------+
|        A|      22|6717736.56|
|        B|      12|1264252.94|
|        C|      12| 475560.69|
+---------+--------+----------+



### 2. 品牌績效排行

In [3]:
brand_returns = returns.join(products.select("product_id", "brand"), "product_id") \
    .groupBy("brand").agg(count("return_id").alias("return_count"), sum("refund_amount").alias("refund_total"))

brand_sold = sales.groupBy("brand").agg(countDistinct("order_id").alias("sold_orders"))

gold_brand = sales.groupBy("brand") \
    .agg(
        sum("line_revenue").alias("total_revenue"),
        sum("line_profit").alias("total_profit"),
        sum("quantity").alias("total_units"),
        countDistinct("order_id").alias("total_orders"),
        countDistinct("user_id").alias("unique_customers")
    ) \
    .join(brand_returns, "brand", "left") \
    .join(brand_sold, "brand", "left") \
    .fillna(0) \
    .withColumn("profit_margin_pct", round(col("total_profit") / col("total_revenue") * 100, 1)) \
    .withColumn("return_rate_pct", round(col("return_count") / col("sold_orders") * 100, 1)) \
    .withColumn("avg_revenue_per_customer", round(col("total_revenue") / col("unique_customers"), 2)) \
    .orderBy(desc("total_revenue"))

print("Brand Performance ready.")
gold_brand.show(5)

Brand Performance ready.
+-------+-------------+------------+-----------+------------+----------------+------------+------------+-----------+-----------------+---------------+------------------------+
|  brand|total_revenue|total_profit|total_units|total_orders|unique_customers|return_count|refund_total|sold_orders|profit_margin_pct|return_rate_pct|avg_revenue_per_customer|
+-------+-------------+------------+-----------+------------+----------------+------------+------------+-----------+-----------------+---------------+------------------------+
|  Apple|   2536258.84|   866740.47|       2528|        1073|             446|          50|    43445.34|       1073|             34.2|            4.7|                 5686.68|
|Samsung|   1016013.37|   399771.06|       1581|         722|             379|          42|    23258.30|        722|             39.3|            5.8|                 2680.77|
| Lenovo|    922922.60|   283058.59|        604|         310|             230|          21|    

### 3. 折扣效果 ROI 分析

In [4]:
order_totals = sales.groupBy("order_id", "discount_amount") \
    .agg(sum("line_revenue").alias("order_revenue"), sum("line_profit").alias("order_profit"),
         sum("quantity").alias("order_items"))

gold_discount = order_totals \
    .withColumn("discount_band",
        when(col("discount_amount") == 0, "No Discount")
        .when(col("discount_amount") <= 50, "$1-50")
        .when(col("discount_amount") <= 100, "$51-100")
        .when(col("discount_amount") <= 150, "$101-150")
        .otherwise("$150+")) \
    .groupBy("discount_band") \
    .agg(
        count("order_id").alias("order_count"),
        round(avg("order_revenue"), 2).alias("avg_order_value"),
        round(avg("order_items"), 1).alias("avg_items_per_order"),
        round(sum("order_revenue"), 2).alias("total_revenue"),
        round(sum("order_profit"), 2).alias("total_profit"),
        round(sum("discount_amount"), 2).alias("total_discount_given")
    ) \
    .withColumn("discount_roi", round(col("total_revenue") / greatest(col("total_discount_given"), lit(1)), 2))

print("Discount ROI ready.")
gold_discount.show()

Discount ROI ready.
+-------------+-----------+---------------+-------------------+-------------+------------+--------------------+------------+
|discount_band|order_count|avg_order_value|avg_items_per_order|total_revenue|total_profit|total_discount_given|discount_roi|
+-------------+-----------+---------------+-------------------+-------------+------------+--------------------+------------+
|  No Discount|       2050|        2889.24|                3.5|   5922936.34|  2072098.53|                0.00|  5922936.34|
|        $150+|        244|        2919.79|                3.5|    712428.60|   242445.44|            42731.77|       16.67|
|        $1-50|        202|        3101.13|                3.7|    626428.57|   218417.06|             4832.80|      129.62|
|      $51-100|        194|        3065.25|                3.8|    594658.66|   202469.21|            14894.24|       39.93|
|     $101-150|        196|        3066.83|                3.6|    601098.02|   204112.39|            246

### 4. 時間模式熱力圖 (星期 x 時段)

In [5]:
gold_hourly = sales \
    .withColumn("hour_of_day", hour("order_date")) \
    .withColumn("day_of_week", date_format("order_date", "EEEE")) \
    .groupBy("day_of_week", "hour_of_day") \
    .agg(
        count("order_id").alias("order_count"),
        sum("line_revenue").alias("total_revenue")
    ) \
    .orderBy("day_of_week", "hour_of_day")

print("Hourly Heatmap ready.")

Hourly Heatmap ready.


### 5. 付款方式分析

In [6]:
gold_payment = sales.groupBy("payment_method") \
    .agg(
        countDistinct("order_id").alias("total_orders"),
        sum("line_revenue").alias("total_revenue"),
        round(avg("line_revenue"), 2).alias("avg_line_value"),
        countDistinct("user_id").alias("unique_users")
    ) \
    .withColumn("revenue_share_pct",
        round(col("total_revenue") / sum("total_revenue").over(Window.partitionBy()), 2) * 100) \
    .orderBy(desc("total_revenue"))

print("Payment Analysis ready.")
gold_payment.show()

Payment Analysis ready.
+----------------+------------+-------------+--------------+------------+-----------------+
|  payment_method|total_orders|total_revenue|avg_line_value|unique_users|revenue_share_pct|
+----------------+------------+-------------+--------------+------------+-----------------+
|     Credit Card|         497|   1502700.77|       1684.64|         317|            18.00|
|Cash on Delivery|         483|   1461982.97|       1705.93|         309|            17.00|
|        LINE Pay|         494|   1429221.75|       1633.40|         320|            17.00|
|       Apple Pay|         473|   1426064.36|       1667.91|         308|            17.00|
|      Debit Card|         454|   1344773.70|       1697.95|         304|            16.00|
|   Bank Transfer|         485|   1292806.64|       1601.99|         317|            15.00|
+----------------+------------+-------------+--------------+------------+-----------------+



### 6. 地區銷售分析

In [7]:
gold_city = sales.join(users.select("user_id", "city"), "user_id") \
    .groupBy("city") \
    .agg(
        countDistinct("user_id").alias("total_customers"),
        countDistinct("order_id").alias("total_orders"),
        sum("line_revenue").alias("total_revenue"),
        sum("line_profit").alias("total_profit"),
        round(avg("line_revenue"), 2).alias("avg_item_value")
    ) \
    .withColumn("revenue_per_customer", round(col("total_revenue") / col("total_customers"), 2)) \
    .orderBy(desc("total_revenue"))

print("City Analysis ready.")
gold_city.show()

City Analysis ready.
+----------+---------------+------------+-------------+------------+--------------+--------------------+
|      city|total_customers|total_orders|total_revenue|total_profit|avg_item_value|revenue_per_customer|
+----------+---------------+------------+-------------+------------+--------------+--------------------+
|  Pingtung|             53|         335|    966436.36|   336873.30|       1600.06|            18234.65|
|   Hsinchu|             56|         318|    932344.49|   327313.55|       1717.02|            16649.01|
|    Tainan|             48|         288|    887630.54|   307149.23|       1640.72|            18492.30|
|  Changhua|             46|         277|    881378.82|   306882.75|       1787.79|            19160.41|
|   Taoyuan|             52|         291|    852664.03|   289125.38|       1665.36|            16397.39|
|  Taichung|             52|         296|    831471.28|   294754.72|       1672.98|            15989.83|
| Kaohsiung|             45|      

### 7. 客戶生命週期價值 (CLV) 預測

In [8]:
current_date_val = sales.agg(max("order_date")).collect()[0][0]

gold_clv = sales.join(users.select("user_id", col("name").alias("user_name"), "city", "member_level", "signup_date"), "user_id") \
    .groupBy("user_id", "user_name", "city", "member_level", "signup_date") \
    .agg(
        countDistinct("order_id").alias("total_orders"),
        sum("line_revenue").alias("total_revenue"),
        sum("line_profit").alias("total_profit"),
        min("order_date").alias("first_purchase"),
        max("order_date").alias("last_purchase"),
        avg("line_revenue").alias("avg_item_value")
    ) \
    .withColumn("customer_age_days", datediff(lit(current_date_val), col("signup_date"))) \
    .withColumn("active_days", datediff(col("last_purchase"), col("first_purchase"))) \
    .withColumn("purchase_frequency", round(col("total_orders") / greatest(col("customer_age_days") / 30, lit(1)), 2)) \
    .withColumn("projected_annual_value",
        round(col("purchase_frequency") * 12 * (col("total_revenue") / greatest(col("total_orders"), lit(1))), 2)) \
    .withColumn("clv_score",
        round(col("projected_annual_value") * least(col("active_days") / 365 + 1, lit(3)), 2)) \
    .withColumn("clv_tier",
        when(col("clv_score") >= 50000, "Platinum")
        .when(col("clv_score") >= 20000, "Gold")
        .when(col("clv_score") >= 5000, "Silver")
        .otherwise("Bronze")) \
    .orderBy(desc("clv_score"))

print("CLV Prediction ready.")
gold_clv.show(5)

CLV Prediction ready.
+-------+----------------+--------+------------+--------------------+------------+-------------+------------+--------------------+--------------------+--------------+-----------------+-----------+------------------+----------------------+---------+--------+
|user_id|       user_name|    city|member_level|         signup_date|total_orders|total_revenue|total_profit|      first_purchase|       last_purchase|avg_item_value|customer_age_days|active_days|purchase_frequency|projected_annual_value|clv_score|clv_tier|
+-------+----------------+--------+------------+--------------------+------------+-------------+------------+--------------------+--------------------+--------------+-----------------+-----------+------------------+----------------------+---------+--------+
|    116|  Rodney Edwards|Pingtung|    Standard|2026-02-17 23:44:...|           7|     32722.99|    12168.44|2026-02-21 09:29:...|2026-03-11 13:23:...|   2181.532667|               22|         18|        

### 寫入所有新 Gold Tables

In [9]:
tables = {
    "gold_abc": gold_abc, "gold_brand": gold_brand, "gold_discount": gold_discount,
    "gold_hourly": gold_hourly, "gold_payment": gold_payment, "gold_city": gold_city, "gold_clv": gold_clv,
}
for name, df in tables.items():
    df.write.jdbc(url=pg_url, table=name, mode="overwrite", properties=pg_props)
    print(f"  [OK] {name}")
print("\n=== ETL v3 Complete! 7 new Gold Tables written. ===")

  [OK] gold_abc
  [OK] gold_brand
  [OK] gold_discount
  [OK] gold_hourly
  [OK] gold_payment
  [OK] gold_city
  [OK] gold_clv

=== ETL v3 Complete! 7 new Gold Tables written. ===
